# Session 08: pandas II: cleaning the mess

Clean the dirty export of Lakeside Mart order lines. Each task has a check. At the end the cleaned net revenue must reconcile to the till system's total, with a named residual.

**SOLUTION VERSION. Tutor only.** Run the cells from the top. Each cell that starts with '# Your turn' is yours to complete. Run it, then run the check cell after it: the check tells you if you are right and what to fix.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, zipfile, urllib.request
DATA_ZIP_URL = ""   # your tutor may give you a link; leave empty to upload data.zip instead
if not os.path.exists("orders.csv"):
    if DATA_ZIP_URL:
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

## Task 1. Load and keep the raw file
Read `orders_raw.csv` into `raw` and print its shape. Make a copy called `df` to clean. The `note` function that fills the cleaning log is given to you.

In [ ]:
import pandas as pd

raw = pd.read_csv('orders_raw.csv')
print(raw.shape)
df = raw.copy()

log = []
def note(step, before, after, why):
    log.append({'step': step, 'rows_before': before, 'rows_after': after, 'why': why})

In [ ]:
assert raw.shape == (41259, 10), 'Check the file name'
assert len(df) == 41259, 'df should be a copy of raw'
print('Task 1 correct')

## Task 2. Duplicates
Drop the exact duplicate lines from `df`. Store the number removed in `removed`, print it, and log the step.

In [ ]:
before = len(df)
df = df.drop_duplicates()
removed = before - len(df)
print('duplicates removed =', removed)
note('Drop exact duplicates', before, len(df), 'identical lines')

In [ ]:
assert removed == 80, 'Use df.drop_duplicates() and compare the lengths'
assert df.line_id.is_unique, 'line_id should now appear once each'
print('Task 2 correct')

## Task 3. Prices and dates
Convert `unit_price` to whole numbers (remove `UGX` and commas first). Then convert `order_date` to real dates: dashes are year-month-day, slashes are day/month/year. Log both steps.

In [ ]:
before = len(df)
price = df.unit_price.str.replace('UGX', '').str.replace(',', '')
df['unit_price'] = price.str.strip().astype(int)
note('Price text to number', before, len(df), 'UGX and commas removed')

iso = pd.to_datetime(df.order_date, format='%Y-%m-%d', errors='coerce')
uk = pd.to_datetime(df.order_date, format='%d/%m/%Y', errors='coerce')
df['order_date'] = iso.fillna(uk)
note('Two date formats to dates', before, len(df), 'slashes read day first')
print(df.unit_price.dtype, df.order_date.dtype)

In [ ]:
assert df.unit_price.dtype == 'int64', 'unit_price should be whole numbers'
assert df.order_date.isna().sum() == 0, 'Every date should have one valid reading'
assert df.order_date.min() == pd.Timestamp('2024-01-01'), 'Earliest date is 1 January 2024'
assert df.order_date.max() == pd.Timestamp('2025-12-28'), 'Latest date is 28 December 2025'
print('Task 3 correct')

## Task 4. Branch and payment text
Trim, fix the case and fix the two typos in `branch` (`Kampala Cntral`, `Entebe`). Make `payment` one of `Mobile money`, `Cash`, `Card`. Log the step.

In [ ]:
before = len(df)
df['branch'] = df.branch.str.strip().str.title()
df['branch'] = df.branch.replace({'Kampala Cntral': 'Kampala Central', 'Entebe': 'Entebbe'})
pay = df.payment.str.strip().str.lower()
df['payment'] = pay.replace({'mm': 'mobile money'}).str.capitalize()
note('Branch and payment text', before, len(df), '26 branch spellings to 8, 11 payment spellings to 3')
print(df.branch.nunique(), df.payment.nunique())

In [ ]:
assert df.branch.nunique() == 8, 'There are eight branches'
assert sorted(df.payment.unique()) == ['Card', 'Cash', 'Mobile money'], 'Three payment methods'
print('Task 4 correct')

## Task 5. Missing customers and missing quantities
Fill every blank customer with `walk-in` (read_csv already made some of them empty). Then drop the lines with no quantity, keeping them in `lost`. Log both steps.

In [ ]:
before = len(df)
df['customer_id'] = df.customer_id.fillna('walk-in')
note('Blank customers to walk-in', before, len(df), 'blank, N/A, NULL and walk-in become one label')
n_walkin = (df.customer_id == 'walk-in').sum()

lost = df[df.qty.isna()]
df = df.dropna(subset=['qty'])
note('Drop blank quantity', before, len(df), '12 lines, quantity cannot be recovered')
print(n_walkin, len(lost), len(df))

In [ ]:
assert n_walkin == 12456, 'Fill customer_id with walk-in and count it'
assert len(lost) == 12, 'Twelve lines have no quantity'
assert len(df) == 41167, 'Drop only the blank quantities, using subset'
print('Task 5 correct')

## Task 6. Outliers: check before you remove
Count the lines whose price differs from `products.csv` (`n_mismatch`), and the lines with a negative quantity (`n_returns`). Remove nothing, and log that.

In [ ]:
prod = pd.read_csv('products.csv')
chk = df.merge(prod, on='product_id')
n_mismatch = (chk.unit_price != chk.unit_price_ugx).sum()
n_returns = (df.qty < 0).sum()
note('Keep returns', len(df), len(df), 'checked and kept, they are real')
print(n_mismatch, n_returns)

In [ ]:
assert n_mismatch == 0, 'Every price should match the price list'
assert n_returns == 1179, 'Count qty below zero'
assert len(df) == 41167, 'Nothing should be removed in this task'
print('Task 6 correct')

## Task 7. Reconcile
Add `revenue = qty * unit_price * (1 - discount_pct / 100)` and sum it into `net`. The till system's own total for the same lines is 561,421,125 UGX, stored as `true_net`. Compute `residual = true_net - net`, then print the cleaning log.

In [ ]:
df['revenue'] = df.qty * df.unit_price * (1 - df.discount_pct / 100)
net = int(round(df.revenue.sum()))
true_net = 561421125
residual = true_net - net
print('net =', net)
print('residual =', residual)
pd.DataFrame(log)

In [ ]:
assert net == 561288435, 'Check the revenue formula and that returns are kept'
assert residual == 132690, 'residual = true_net - net'
assert len(log) == 7, 'The log should have 7 rows'
assert log[-1]['rows_after'] == 41167, 'Last row count should be 41167'
print('Task 7 correct')

## Task 8. Name the residual
In one sentence in this cell, say what the residual is and what causes it. Then save your work with `df.to_csv('orders_clean.csv', index=False)` and write one line on how you would explain the cleaning log to a manager. This notebook is Checkpoint 1: it passes when every check passes and the residual is named.